# Hyperspectral ViT — training

Set `DATASET_ROOT` to a directory built with `download_dataset.run_full_dataset_build` (contains `dataset_index.csv` and `instrument.json`). For a dry run without EMIT chips, set `USE_SYNTHETIC_DATA = True`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

# encoder_refactor on path
_ROOT = Path("/workspace/Hyperspectral_Coolstuff/encoder_refactor").resolve()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import numpy as np
import torch
from torch.utils.data import DataLoader

from utilities import (
    EmitHypercubeIndexDataset,
    collate_hypercube_batch,
    linear_warmup_cosine_scheduler,
    load_band_stats_npz,
    load_dataset_index_csv,
    masked_band_mse,
    num_bands_from_instrument,
    save_checkpoint,
    set_seed,
    train_val_split_by_granule,
)
from model_definition import ModelConfig, build_model, model_config_to_dict

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", DEVICE)

In [ ]:
# --- User configuration ---------------------------------------------------

DATASET_ROOT = Path("/workspace/Hyperspectral_Coolstuff/emit_dataset")  # change me
INDEX_CSV = DATASET_ROOT / "dataset_index.csv"
BAND_STATS_NPZ = DATASET_ROOT / "emit_band_stats.npz"  # optional; omit normalization if missing

USE_SYNTHETIC_DATA = not INDEX_CSV.is_file()

SPATIAL_SIZE = (256, 256)
IN_CHANNELS = None  # filled from instrument.json when using real data
BATCH_SIZE = 2
NUM_EPOCHS = 3
LR = 3e-4
WEIGHT_DECAY = 0.05
SEED = 42
VAL_FRACTION = 0.15
CHECKPOINT_PATH = _ROOT / "checkpoints" / "vit_hs_latest.pt"

set_seed(SEED)

In [ ]:
from torch.utils.data import Dataset


class SyntheticCubeDataset(Dataset):
    """Random NCHW cubes for smoke-training when no dataset_index.csv exists."""

    def __init__(self, n: int, channels: int, spatial: tuple[int, int]):
        self.n = n
        self.channels = channels
        self.spatial = spatial

    def __len__(self):
        return self.n

    def __getitem__(self, idx: int):
        h, w = self.spatial
        x = torch.randn(self.channels, h, w)
        valid = torch.ones_like(x)
        meta = {"id": f"synth_{idx}", "label": "synthetic"}
        return {"cube": x, "valid": valid, "meta": meta}


def collate_synth(samples):
    cubes = torch.stack([s["cube"] for s in samples])
    valids = torch.stack([s["valid"] for s in samples])
    return {"cube": cubes, "valid": valids, "meta": [s["meta"] for s in samples]}


if USE_SYNTHETIC_DATA:
    print("Using synthetic cubes (no dataset_index.csv at", INDEX_CSV, ")")
    IN_CHANNELS = 32
    train_ds = SyntheticCubeDataset(64, IN_CHANNELS, SPATIAL_SIZE)
    val_ds = SyntheticCubeDataset(16, IN_CHANNELS, SPATIAL_SIZE)
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_synth, num_workers=0
    )
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_synth, num_workers=0)
else:
    IN_CHANNELS = num_bands_from_instrument(DATASET_ROOT)
    df = load_dataset_index_csv(INDEX_CSV)
    df_train, df_val = train_val_split_by_granule(df, VAL_FRACTION, SEED)
    band_mean = band_std = None
    if BAND_STATS_NPZ.is_file():
        band_mean, band_std = load_band_stats_npz(BAND_STATS_NPZ)
    train_ds = EmitHypercubeIndexDataset(
        df_train, DATASET_ROOT, band_mean=band_mean, band_std=band_std
    )
    val_ds = EmitHypercubeIndexDataset(df_val, DATASET_ROOT, band_mean=band_mean, band_std=band_std)
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_hypercube_batch,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_hypercube_batch,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )

print("train batches", len(train_loader), "val batches", len(val_loader), "C", IN_CHANNELS)

In [ ]:
# Patch geometry must tile SPATIAL_SIZE
PATCH_H, PATCH_W = 16, 16

model_cfg = ModelConfig(
    spatial_size=SPATIAL_SIZE,
    in_channels=IN_CHANNELS,
    embed_dim=128,
    depth=4,
    num_heads=8,
    mlp_ratio=4.0,
    dropout=0.0,
    tokenizer_type="patch2d",
    tokenizer_kwargs={"patch_h": PATCH_H, "patch_w": PATCH_W},
    pos_encoding_type="learned",
    backbone_type="preln_vit",
    head_type="cube_fold_patch2d",
    head_kwargs={},
    use_cls_token=False,
)

model = build_model(model_cfg).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

total_steps = max(1, NUM_EPOCHS * len(train_loader))
warmup_steps = min(500, total_steps // 10)
scheduler = linear_warmup_cosine_scheduler(optimizer, warmup_steps, total_steps)

print(model_cfg)

In [ ]:
def train_one_epoch():
    model.train()
    losses = []
    for batch in train_loader:
        x = batch["cube"].to(DEVICE)
        valid = batch["valid"].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        out = model(x)
        pred = out.cube_reconstruction
        if pred is None:
            raise RuntimeError("Model has no reconstruction head; pick head_type != identity")
        loss = masked_band_mse(pred, x, valid)
        loss.backward()
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())
    return float(np.mean(losses))


@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for batch in val_loader:
        x = batch["cube"].to(DEVICE)
        valid = batch["valid"].to(DEVICE)
        out = model(x)
        pred = out.cube_reconstruction
        losses.append(masked_band_mse(pred, x, valid).item())
    return float(np.mean(losses))


best_val = float("inf")
for epoch in range(NUM_EPOCHS):
    tr = train_one_epoch()
    va = evaluate()
    print(f"epoch {epoch+1}/{NUM_EPOCHS}  train {tr:.6f}  val {va:.6f}")
    if va < best_val:
        best_val = va
        CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
        save_checkpoint(
            CHECKPOINT_PATH,
            model_state_dict=model.state_dict(),
            model_config_dict=model_config_to_dict(model_cfg),
            optimizer_state_dict=optimizer.state_dict(),
            epoch=epoch,
            extra={"best_val": best_val},
        )
        print("  saved", CHECKPOINT_PATH)

print("done. best val", best_val)